# Weaver: from disorganized to organized complexity

This is a deliberately simple **desire-path** simulation.

We use exactly the same:

- people,
- space,
- destinations,
- movement possibilities.

We compare only two situations:

1. **No memory / no feedback:** walkers choose independently among reasonable steps toward their destination.
2. **Memory + feedback:** walkers can see which places were used before and tend to follow them.

The question is simple:

> What changes when past actions begin to organize future actions?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# World
WIDTH = 41
HEIGHT = 31

# Three destinations
buildings = [
    (3, 5),
    (3, 25),
    (37, 15)
]


## The model

Each walker repeatedly travels from one building to another.

At every step, the walker considers only neighboring cells that take it **closer to its destination**.

- With **memory OFF**, it chooses randomly among those cells.
- With **memory ON**, it usually chooses the most popular available cell.

Every visit increases a cell's `popularity`.

That is the only difference between the two simulations.


In [ ]:
def closer_neighbors(x, y, gx, gy):
    current_distance = (x - gx)**2 + (y - gy)**2

    candidates = []

    for dx, dy in [
        (1, 0), (-1, 0), (0, 1), (0, -1),
        (1, 1), (1, -1), (-1, 1), (-1, -1)
    ]:
        nx = x + dx
        ny = y + dy

        if 0 <= nx < WIDTH and 0 <= ny < HEIGHT:
            new_distance = (nx - gx)**2 + (ny - gy)**2

            if new_distance < current_distance:
                candidates.append((nx, ny))

    return candidates


In [ ]:
def run_paths(memory=False, seed=123,
              n_walkers=80,
              total_steps=20000,
              follow_probability=0.85):

    rng = np.random.default_rng(seed)

    popularity = np.zeros((HEIGHT, WIDTH), dtype=int)

    positions = []
    goals = []

    # Every walker starts at one building
    # and receives another building as its destination.
    for _ in range(n_walkers):
        start, goal = rng.choice(
            len(buildings),
            size=2,
            replace=False
        )

        positions.append(buildings[start])
        goals.append(buildings[goal])

    steps = 0

    while steps < total_steps:

        for i in range(n_walkers):

            x, y = positions[i]
            gx, gy = goals[i]

            # Arrived: choose another building
            if (x, y) == (gx, gy):
                current = buildings.index((gx, gy))

                possible_goals = [
                    j for j in range(len(buildings))
                    if j != current
                ]

                new_goal = rng.choice(possible_goals)
                goals[i] = buildings[new_goal]

                continue

            candidates = closer_neighbors(x, y, gx, gy)

            # Defensive guard: with this grid and these building positions this
            # never actually happens (checked empirically), but if you move a
            # building near the edge of the grid later, a walker could in
            # principle find zero "closer" neighbors. Rather than crash, it
            # just sits still for this tick.
            if len(candidates) == 0:
                continue

            # MEMORY OFF:
            # choose independently among reasonable steps.
            if not memory:
                nx, ny = candidates[
                    rng.integers(len(candidates))
                ]

            # MEMORY ON:
            # usually follow the most-used available cell.
            else:
                if rng.random() < follow_probability:

                    values = [
                        popularity[cy, cx]
                        for cx, cy in candidates
                    ]

                    best_value = max(values)

                    best_cells = [
                        cell for cell, value
                        in zip(candidates, values)
                        if value == best_value
                    ]

                    nx, ny = best_cells[
                        rng.integers(len(best_cells))
                    ]

                else:
                    nx, ny = candidates[
                        rng.integers(len(candidates))
                    ]

            positions[i] = (nx, ny)

            popularity[ny, nx] += 1

            steps += 1

            if steps >= total_steps:
                break

    return popularity


## Case A — No memory

Walkers have the same destinations and the same possible movements, but each movement decision is independent of previous walkers.

Past movement leaves no information that later walkers use.


In [ ]:
no_memory = run_paths(
    memory=False,
    seed=123
)

plt.figure(figsize=(8, 6))
plt.imshow(no_memory, origin="lower")

for x, y in buildings:
    plt.scatter(x, y, marker="s", s=80)

plt.title("No memory: cumulative foot traffic")
plt.xlabel("x")
plt.ylabel("y")
plt.show()


## Case B — Memory and feedback

Now walkers can respond to the history of the system.

Frequently used places become more attractive to later walkers.

So:

**movement → popularity → later movement → more popularity**


In [ ]:
with_memory = run_paths(
    memory=True,
    seed=123
)

plt.figure(figsize=(8, 6))
plt.imshow(with_memory, origin="lower")

for x, y in buildings:
    plt.scatter(x, y, marker="s", s=80)

plt.title("Memory + feedback: cumulative foot traffic")
plt.xlabel("x")
plt.ylabel("y")
plt.show()


## A very simple comparison

If movement becomes organized around a smaller number of cells, a greater share of all footsteps should be concentrated in the most-used parts of the landscape.

We measure the share of all footsteps occurring in the **top 5% most-used cells**.


In [ ]:
def top_share(popularity, fraction=0.05):
    values = np.sort(popularity.ravel())[::-1]

    n_top = max(
        1,
        int(len(values) * fraction)
    )

    return values[:n_top].sum() / values.sum()


print(
    "No memory:",
    round(top_share(no_memory), 3)
)

print(
    "Memory + feedback:",
    round(top_share(with_memory), 3)
)


## Does the pattern repeat, or was that just one lucky run?

So far we've compared exactly **one** run with memory off to exactly **one** run with memory on. That tells us the two conditions differ -- but not yet *what kind* of difference this is.

This is really the crux of Weaver's distinction. It's not "no memory = boring, memory = interesting." It's:

- **Disorganized complexity**: individual behavior is unpredictable, but if you re-run the whole thing with different randomness, the *group-level result* comes out almost identical every time.
- **Organized complexity**: the group-level result *itself* stops being reliably predictable, because the system has memory -- whatever got a head start early on shapes everything that follows.

So let's actually re-run both conditions several times, changing nothing but the random seed, and see which kind of difference we're looking at.


In [ ]:
seeds_to_try = [1, 2, 3, 4, 5]

no_memory_shares = []
with_memory_shares = []

for seed in seeds_to_try:
    pop_no_memory = run_paths(memory=False, seed=seed)
    pop_with_memory = run_paths(memory=True, seed=seed)

    no_memory_shares.append(top_share(pop_no_memory))
    with_memory_shares.append(top_share(pop_with_memory))

print("No memory, top-5% share across 5 different runs:")
print(" ", [round(s, 3) for s in no_memory_shares])

print()
print("Memory + feedback, top-5% share across 5 different runs:")
print(" ", [round(s, 3) for s in with_memory_shares])


In [ ]:
def spread_pct(values):
    values = np.array(values)
    return 100 * (values.max() - values.min()) / values.mean()

print(f"No memory:          the top-5% share varies by {spread_pct(no_memory_shares):.1f}% across runs.")
print(f"Memory + feedback:   the top-5% share varies by {spread_pct(with_memory_shares):.1f}% across runs.")


That's already a real difference -- but it understates the story. Two runs could land on a *similar number* while still being organized around **completely different cells**. Let's check that directly: for the memory condition, does each run even agree on *which* cells are the popular ones?


In [ ]:
def top_cells(popularity, fraction=0.05):
    flat_ranked = np.argsort(popularity.ravel())[::-1]
    n_top = max(1, int(popularity.size * fraction))
    return set(flat_ranked[:n_top])

memory_top_cells = [top_cells(run_paths(memory=True, seed=seed)) for seed in seeds_to_try]

print("How much do different runs agree on WHICH exact cells ended up in the top 5%?")
for i in range(len(seeds_to_try)):
    for j in range(i + 1, len(seeds_to_try)):
        overlap = len(memory_top_cells[i] & memory_top_cells[j]) / len(memory_top_cells[i] | memory_top_cells[j])
        print(f"  seed {seeds_to_try[i]} vs seed {seeds_to_try[j]}: {overlap*100:.0f}% overlap")


**What this adds:** the *share* number (how concentrated traffic gets) is only moderately more variable with memory on than off. But *which specific cells* become the popular ones barely repeats at all between runs -- typically less than 20% overlap. Even the walkers who start and end at the exact same three buildings, using the exact same rule, end up organizing themselves around a different specific layout almost every time.

That's the real signature of organized complexity here: not "more variable," but **historically path-dependent**. Whatever cell happened to get a slight popularity edge early on -- pure luck, from the very first walkers -- shapes the whole rest of the run. No formula predicts in advance which cells those will be; you have to run it and watch.


# What should we notice?

In both simulations:

- there are the same walkers;
- they move in the same world;
- they travel among the same destinations;
- they have the same possible steps.

The difference is **memory and feedback**.

### Without memory

A walker's action does not organize the choices of later walkers.

Many individual movements occur, but their histories do not become part of the system.

And when we reran this condition with different random seeds, the group-level result (the top-5% share) barely moved -- a strong hint that this is closer to disorganized complexity, in Weaver's sense: individuals are unpredictable, but the aggregate isn't.

### With memory

Past movements change the information available to later walkers.

Repeated use reinforces some routes:

**past behavior → structure → future behavior**

And when we reran *this* condition with different random seeds, not only did the group-level number move more -- the *specific* cells that became popular were almost entirely different each time. The system has memory, so early, essentially random events get locked in and shape everything after them. That's not just "more complicated" than the no-memory case; it's a different *kind* of unpredictability, one that no averaging trick removes.

This provides a simple visual analogy for Weaver's distinction.

> Organized complexity is not simply about having many elements.
> The relationships among actions — including feedback through structures created by earlier actions — can become essential for understanding the behavior of the whole.

### Important

This is a teaching illustration, not a claim that every system with feedback is automatically a complete example of Weaver's organized complexity.

One caveat worth being explicit about: "no memory" here isn't the same as a truly disorganized (helter-skelter) baseline in Weaver's strict sense, either. All walkers in *both* conditions still share the same 3 fixed destinations, so some concentration around the direct paths between buildings exists even with memory off -- what memory adds on top of that is the *self-reinforcing, history-dependent* part specifically, not organization from nothing.

(If you're curious what this mechanism is called elsewhere: a rule like "the more a cell has been used, the more likely it is to be used again" is a version of what mathematicians call a **Pólya urn** process -- a classic example of a system whose long-run outcome is random but far from uniform, and which typically converges to a *different* answer nearly every time you run it.)
